#  02 — Data Cleaning Pipeline
### Forecasting & Identifying Global Export Opportunities for Algerian Exporters
**ENSIA — Machine Learning Project | Spring 2025–2026**

---

This notebook cleans all raw tables produced by `01_data_collection.ipynb`.

| # | Input file (`data/raw/`) | Output file (`data/processed/`) | What is cleaned |
|---|--------------------------|----------------------------------|-----------------|
| 1 | `01_comtrade_world_imports.csv` | `01_comtrade_world_imports_clean.csv` | Remove World rows, drop noise columns, fill missing values |
| 2 | `02_algeria_resources_fao.csv` | `02_algeria_resources_fao_clean.csv` | Drop metadata columns |
| 3 | `03_algeria_resources_fao_inputs.csv` | `03_algeria_resources_fao_inputs_clean.csv` | Drop metadata columns |
| 4 | `04_algeria_resources_fao_land.csv` | `04_algeria_resources_fao_land_clean.csv` | Drop metadata columns |
| 5 | `05_algeria_resources_fao_prices.csv` | `05_algeria_resources_fao_prices_clean.csv` | Drop metadata columns |
| 6 | `06_algeria_resources_worldbank.csv` | `06_algeria_resources_worldbank_clean.csv` | Fill missing values with median |

**Reading order:** Run cells top to bottom. Each section is independent.


---
## ⚙️ Setup


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure output folder exists
Path("data/processed").mkdir(parents=True, exist_ok=True)

print(" Ready — input:  data/raw/")
print("          output: data/processed/")


---
##  Table 1 — UN Comtrade: Global Import Flows
**Input:**  `data/raw/01_comtrade_world_imports.csv`  
**Output:** `data/processed/01_comtrade_world_imports_clean.csv`

### Cleaning steps:
1. **Remove "World" exporter rows** — these are aggregate rows, not real bilateral trades
2. **Drop low-value columns** — `data_source`, `aggregate_level`, `is_leaf_code`, `mode_of_transport_code`
3. **Fill missing values:**
   - Numerical → median (robust to extreme outliers in trade data)
   - Categorical → most frequent value (mode)


In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv("data/raw/01_comtrade_world_imports.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ── Step 1: Remove aggregate "World" exporter rows ───────────────────────────
df = df[df["exporter_name"] != "World"]
print(f"After removing 'World' rows: {df.shape[0]:,} rows")

# ── Step 2: Drop noise/metadata columns ──────────────────────────────────────
cols_to_drop = ["data_source", "aggregate_level", "is_leaf_code", "mode_of_transport_code"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"After dropping metadata columns: {df.shape[1]} columns remaining")

# ── Step 3: Check missing values ─────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("\nNo missing values found.")
else:
    print("\nMissing values before filling:")
    print(missing)

# ── Step 4: Fill missing values ───────────────────────────────────────────────
print("\nNumerical column stats (to understand outlier scale):")
print(df.select_dtypes(include=["float64", "int64"]).describe().round(2))

for col in df.columns:
    if df[col].isnull().sum() == 0:
        continue
    if df[col].dtype in ["float64", "int64"]:
        fill_val = df[col].median()
        df[col] = df[col].fillna(fill_val)
        print(f"  [Numerical]   '{col}' → filled with median: {fill_val:.4f}")
    else:
        fill_val = df[col].mode()[0]
        df[col] = df[col].fillna(fill_val)
        print(f"  [Categorical] '{col}' → filled with mode: '{fill_val}'")

# ── Verify ─────────────────────────────────────────────────────────────────────
remaining_missing = df.isnull().sum().sum()
print(f"\nMissing values after cleaning: {remaining_missing}")
print(f"Final shape: {df.shape}")

# ── Save ───────────────────────────────────────────────────────────────────────
df.to_csv("data/processed/01_comtrade_world_imports_clean.csv", index=False)
print("\n Saved → data/processed/01_comtrade_world_imports_clean.csv")
df.head(3)


---
## 🔹 Table 2 — FAO: Algeria Agricultural Production
**Input:**  `data/raw/02_algeria_resources_fao.csv`  
**Output:** `data/processed/02_algeria_resources_fao_clean.csv`

### Cleaning steps:
1. **Drop metadata columns** — `Note`, `data_source`
   (these are FAO internal flags not needed for analysis)


In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv("data/raw/02_algeria_resources_fao.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

# ── Drop metadata columns ─────────────────────────────────────────────────────
cols_to_drop = ["Note", "data_source"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"\nAfter dropping metadata: {df.shape[1]} columns remaining")

# ── Check missing values ──────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"Missing values: {missing.to_dict() if not missing.empty else 'none'}")

# ── Save ─────────────────────────────────────────────────────────────────────
df.to_csv("data/processed/02_algeria_resources_fao_clean.csv", index=False)
print("\n Saved → data/processed/02_algeria_resources_fao_clean.csv")
df.head(3)


---
##  Table 3 — FAO: Algeria Agricultural Inputs
**Input:**  `data/raw/03_algeria_resources_fao_inputs.csv`  
**Output:** `data/processed/03_algeria_resources_fao_inputs_clean.csv`

### Cleaning steps:
1. **Drop metadata columns** — `Note`, `hs_code_6digit`, `data_source`
   (`hs_code_6digit` is not meaningful for fertilizer/input data)


In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv("data/raw/03_algeria_resources_fao_inputs.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

# ── Drop metadata columns ─────────────────────────────────────────────────────
cols_to_drop = ["Note", "hs_code_6digit", "data_source"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"\nAfter dropping metadata: {df.shape[1]} columns remaining")

# ── Check missing values ──────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"Missing values: {missing.to_dict() if not missing.empty else 'none'}")

# ── Save ─────────────────────────────────────────────────────────────────────
df.to_csv("data/processed/03_algeria_resources_fao_inputs_clean.csv", index=False)
print("\n Saved → data/processed/03_algeria_resources_fao_inputs_clean.csv")
df.head(3)


---
##  Table 4 — FAO: Algeria Land Use & Irrigation
**Input:**  `data/raw/04_algeria_resources_fao_land.csv`  
**Output:** `data/processed/04_algeria_resources_fao_land_clean.csv`

### Cleaning steps:
1. **Drop metadata columns** — `Note`, `hs_code_6digit`, `data_source`
   (land use data has no meaningful HS product code mapping)


In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv("data/raw/04_algeria_resources_fao_land.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

# ── Drop metadata columns ─────────────────────────────────────────────────────
cols_to_drop = ["Note", "hs_code_6digit", "data_source"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"\nAfter dropping metadata: {df.shape[1]} columns remaining")

# ── Check missing values ──────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"Missing values: {missing.to_dict() if not missing.empty else 'none'}")

# ── Save ─────────────────────────────────────────────────────────────────────
df.to_csv("data/processed/04_algeria_resources_fao_land_clean.csv", index=False)
print("\n Saved → data/processed/04_algeria_resources_fao_land_clean.csv")
df.head(3)


---
##  Table 5 — FAO: Algeria Domestic Producer Prices
**Input:**  `data/raw/05_algeria_resources_fao_prices.csv`  
**Output:** `data/processed/05_algeria_resources_fao_prices_clean.csv`

### Cleaning steps:
1. **Drop metadata columns** — `Unit`, `hs_code_6digit`, `data_source`
   (`Unit` is always USD/tonne for this dataset — redundant)


In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv("data/raw/05_algeria_resources_fao_prices.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

# ── Drop metadata columns ─────────────────────────────────────────────────────
cols_to_drop = ["Unit", "hs_code_6digit", "data_source"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"\nAfter dropping metadata: {df.shape[1]} columns remaining")

# ── Check missing values ──────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"Missing values: {missing.to_dict() if not missing.empty else 'none'}")

# ── Save ─────────────────────────────────────────────────────────────────────
df.to_csv("data/processed/05_algeria_resources_fao_prices_clean.csv", index=False)
print("\n Saved → data/processed/05_algeria_resources_fao_prices_clean.csv")
df.head(3)


---
##  Table 6 — World Bank: Algeria Resource Indicators
**Input:**  `data/raw/06_algeria_resources_worldbank.csv`  
**Output:** `data/processed/06_algeria_resources_worldbank_clean.csv`

### Cleaning steps:
1. **Drop `data_source`** — not needed for analysis
2. **Fill missing `value` with median** — some indicators have gaps for certain years;
   median is preferred over mean because GDP/rent indicators can be heavily skewed


In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv("data/raw/06_algeria_resources_worldbank.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

# ── Step 1: Drop metadata column ──────────────────────────────────────────────
df = df.drop(columns=["data_source"], errors="ignore")

# ── Step 2: Show missing values ───────────────────────────────────────────────
n_missing = df["value"].isnull().sum()
print(f"\nMissing values in 'value' column: {n_missing}")

# ── Step 3: Fill with median ──────────────────────────────────────────────────
value_median = df["value"].median()
df["value"] = df["value"].fillna(value_median)
print(f"'value' → filled with median: {value_median:.4f}")

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"\nMissing values after cleaning: {df['value'].isnull().sum()}")
print(f"Final shape: {df.shape}")

# ── Save ─────────────────────────────────────────────────────────────────────
df.to_csv("data/processed/06_algeria_resources_worldbank_clean.csv", index=False)
print("\n Saved → data/processed/06_algeria_resources_worldbank_clean.csv")
df.head(3)


---
##  Summary — Check All Cleaned Files


In [ ]:
import os

files = [
    ("data/raw/01_comtrade_world_imports.csv",          "data/processed/01_comtrade_world_imports_clean.csv"),
    ("data/raw/02_algeria_resources_fao.csv",           "data/processed/02_algeria_resources_fao_clean.csv"),
    ("data/raw/03_algeria_resources_fao_inputs.csv",    "data/processed/03_algeria_resources_fao_inputs_clean.csv"),
    ("data/raw/04_algeria_resources_fao_land.csv",      "data/processed/04_algeria_resources_fao_land_clean.csv"),
    ("data/raw/05_algeria_resources_fao_prices.csv",    "data/processed/05_algeria_resources_fao_prices_clean.csv"),
    ("data/raw/06_algeria_resources_worldbank.csv",     "data/processed/06_algeria_resources_worldbank_clean.csv"),
]

print("=" * 80)
print(f"  {'File':<50}  {'Raw rows':>10}  {'Clean rows':>10}")
print("=" * 80)

all_ok = True
for raw_path, clean_path in files:
    def row_count(p):
        if not os.path.exists(p): return -1
        try: return sum(1 for _ in open(p, encoding="utf-8", errors="ignore")) - 1
        except: return -1

    raw_rows   = row_count(raw_path)
    clean_rows = row_count(clean_path)
    icon = "yes" if clean_rows > 0 else "no"
    if clean_rows <= 0: all_ok = False

    name = os.path.basename(clean_path).replace("_clean.csv", "")
    print(f" {icon}  {name:<50}  {raw_rows:>10,}  {clean_rows:>10,}")

print("=" * 80)
if all_ok:
    print("\n All 6 tables cleaned successfully!")
    print("   Next step: open 02_data_preparation_eda.ipynb for EDA and feature engineering")
else:
    print("\n⚠  Some output files are missing. Re-run the failed cells above.")
